[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivastar/ml_for_astro_notebooks/blob/main/session2a_student.ipynb)

# Session 2 — Unsupervised Learning: Clustering

The regression problem yesterday had **labels**: a redshift to regress on. Another example where we may have some labels is a classification problem where some of the objects are known to belong to a class as we will see tomorrow. Most of the time in astronomy we have no labels — only a large catalogue and the suspicion that it contains distinct populations: stellar streams in phase space, galaxy populations in colour space, spectral classes, groups in a redshift survey.

**Clustering** asks: *can we find groups in the data without being told what they are?* This raises three questions, which structure this session:

1. **How** do we find clusters? (the algorithm, and what it implicitly assumes)
2. **How many** clusters are there? (choosing $k$)
3. **Are they real?** (cluster validation)

We work with **simulated data** throughout, because only then do we know the right answer and can check how well each method recovers it.

Cells marked **✏️ Exercise** are for you to complete.

In [ ]:
import sys
if "google.colab" in sys.modules:
    !pip install -q -U scikit-learn

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, DBSCAN, HDBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (silhouette_score, silhouette_samples,
                             adjusted_rand_score, normalized_mutual_info_score)

In [ ]:
#@title Setup (run me) {display-mode: "form"}

plt.rcParams.update({
    "figure.figsize": (7, 5), "figure.dpi": 110, "font.size": 12, "axes.titlesize": 13,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "lines.linewidth": 2, "legend.frameon": False,
})

def plot_clusters(X, labels, ax, title="", centers=None, s=10):
    # colour points by cluster label (matplotlib's default cycle); -1 (noise) is light grey
    labels = np.asarray(labels)
    for k in np.unique(labels):
        m = labels == k
        col = "lightgray" if k == -1 else f"C{k % 10}"
        ax.scatter(X[m, 0], X[m, 1], s=s, color=col, alpha=0.75, linewidth=0,
                   label="noise" if k == -1 else None)
    if centers is not None:
        ax.scatter(centers[:, 0], centers[:, 1], marker="X", s=160, color="black",
                   edgecolor="white", linewidth=1.5, zorder=5)
    ax.set(title=title, xlabel="feature 1", ylabel="feature 2")
    ax.set_aspect("equal", adjustable="datalim")

## The data

Four groups in two dimensions, with different sizes and spreads. Think of them as, e.g., four populations in a colour–magnitude or phase-space diagram. We keep the true labels `y_true` **only for evaluation** — the clustering algorithms never see them.

In [ ]:
X, y_true = make_blobs(n_samples=[400, 300, 250, 150],
                       centers=[(-4, 2), (1, 4), (2, -1), (-2, -3)],
                       cluster_std=[1.3, 0.8, 1.0, 0.6], random_state=11)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.scatter(X[:, 0], X[:, 1], s=10, color="gray", alpha=0.6, linewidth=0)
ax1.set(title="What the algorithm sees", xlabel="feature 1", ylabel="feature 2")
ax1.set_aspect("equal", adjustable="datalim")
plot_clusters(X, y_true, ax2, "The truth (hidden)");

## 1. $k$-means

$k$-means looks for $k$ cluster centres $\boldsymbol\mu_1, \dots, \boldsymbol\mu_k$ and an assignment of every point to one centre that minimises the **within-cluster sum of squares** (also called *inertia*):

$$J = \sum_{j=1}^{k} \sum_{i \in S_j} \lVert \mathbf{x}_i - \boldsymbol\mu_j \rVert^2 .$$

Finding the global minimum is difficult one can find *local minima* by alternating 2 steps each of which can only decrease $J$ (a.k.a., Lloyd's algorithm):

1. **Assign** each point to its nearest centre.
2. **Update** each centre to the *mean* of the points assigned to it.

Repeat until the assignments stop changing. The result depends on the **initial centres**, so in practice the algorithm is run many times from different starts (and `scikit-learn` uses the smarter *k-means++* initialisation).

### ✏️ Exercise 1 — The algorithm

Complete the two steps inside the loop. The function returns the labels, the centres, and the value of $J$ after every iteration.

*Hint:* `np.linalg.norm(X[:, None, :] - centers[None, :, :], axis=2)` gives the $(N, k)$ matrix of distances between every point and every centre.

In [ ]:
def kmeans(X, k, n_iter=50, seed=0):
    rng = np.random.default_rng(seed)
    centers = X[rng.choice(len(X), size=k, replace=False)]      # random initial centres
    history = []
    for it in range(n_iter):
        # YOUR CODE HERE, should define labels (nearest-centre assignment) and new_centers (mean of each group)
        # dist =
        # labels =
        # new_centers =
        history.append(np.sum((X - new_centers[labels])**2))
        if np.allclose(new_centers, centers):
            break
        centers = new_centers
    return labels, centers, np.array(history)

labels_4, centers_4, J_hist = kmeans(X, k=4, seed=1)
print(f"converged after {len(J_hist)} iterations, J = {J_hist[-1]:.1f}")

In [ ]:
# Compare against scikit-learn, and look at several random starts
fig, axes = plt.subplots(1, 4, figsize=(18, 4.4))
axes[0].plot(np.arange(1, len(J_hist) + 1), J_hist, "o-", ms=5)
axes[0].set(xlabel="iteration", ylabel="J (inertia)", title="J never increases")
axes[0].set_aspect("auto")

for ax, seed in zip(axes[1:], [0, 1, 6]):
    lab, cen, hist = kmeans(X, k=4, seed=seed)
    plot_clusters(X, lab, ax, f"random start {seed}:  J = {hist[-1]:.0f}", centers=cen, s=6)

fig.tight_layout();

In [ ]:
# we can also do the same with sklearn:
km = KMeans(n_clusters=4, n_init=10, random_state=0).fit(X)
print(f"scikit-learn KMeans (k-means++, 10 starts): J = {km.inertia_:.1f}")

fig, (ax1) = plt.subplots(1, 1, figsize=(6, 5))
plot_clusters(X, km.labels_, ax1, title="scikit-learn KMeans")

## 2. Choosing $k$

$k$-means needs $k$ as input. Can the data tell us?

**The elbow method.** $J$ — the inertia that $k$-means actually minimises — always decreases with $k$ (with $k = N$, $J = 0$). Look for the "elbow" where adding clusters stops helping much. Simple, but the elbow is often ambiguous.

**The silhouette score.** For each point $i$, define two plain averages:
* **Cohesion** $a(i)$: the average distance from $i$ to the other points in its *own* cluster. Lower values mean a tighter, better-formed cluster.
* **Separation** $b(i)$: the average distance from $i$ to the points in the *nearest neighbouring* cluster. Higher values mean the clusters are far apart.

Then

$$s(i) = \frac{b(i) - a(i)}{\max\{a(i),\, b(i)\}} \in [-1, 1] .$$

$s \approx 1$: well inside its cluster; $s \approx 0$: on a boundary; $s < 0$: probably in the wrong cluster. The mean silhouette over all points is a single number we can maximise over $k$. `scikit-learn` computes all of this for us (`silhouette_score`, `silhouette_samples`) — you don't need to code $a(i)$ or $b(i)$ by hand.

*If silhouette is the better diagnostic, why bother with inertia at all?* Because inertia is $k$-means' actual objective — it's what the algorithm minimises, so the elbow tells you about the optimisation itself. Silhouette is a separate, after-the-fact measure of how well-separated the *result* is, and the two don't always agree.

**Information criteria.** If we have a *probabilistic* model — for example a mixture of Gaussians — we can compute the likelihood, and penalise the number of parameters $p$ with the **Bayesian Information Criterion**

$$\mathrm{BIC} = -2 \ln \hat L + p \ln N ,$$

and choose the $k$ with the **lowest** BIC. $\hat L$ is the model's best-fit likelihood of the data (how well it fits), and $p$ is its number of free parameters (means, covariances and weights) — more components always fit better, but BIC charges a price for the extra parameters needed to get there.

### ✏️ Exercise 2 — Elbow and silhouette

For $k = 2, \dots, 10$, fit `KMeans(n_clusters=k, n_init=10, random_state=0)` and store the inertia in `inertias` and the mean silhouette score in `silhouettes`. Which $k$ does each method suggest?

In [ ]:
ks = np.arange(2, 11)
inertias, silhouettes = [], []

# YOUR CODE HERE, for each k fit KMeans(n_clusters=k, n_init=10, random_state=0) and append
# its inertia to inertias and its silhouette score to silhouettes
# for k in ks:
#     km_k =
#     inertias.append(...)
#     silhouettes.append(...)

k_sil = ks[np.argmax(silhouettes)]
print(f"best k by silhouette: {k_sil}")

### ✏️ Exercise 3 — BIC for a Gaussian mixture

Fit `GaussianMixture(n_components=k, covariance_type="full", n_init=3, random_state=0)` for the same range of $k$ and store `.bic(X)` in `bics`. The next cell plots all three criteria side by side.

In [ ]:
bics = []

# YOUR CODE HERE, for each k fit GaussianMixture(n_components=k, covariance_type="full", n_init=3, random_state=0)
# and append its .bic(X) to bics
# for k in ks:
#     gmm_k =
#     bics.append(...)

k_bic = ks[np.argmin(bics)]
print(f"best k by BIC: {k_bic}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, vals, lab, best in zip(
        axes, [inertias, silhouettes, bics],
        ["inertia J (k-means)", "mean silhouette (k-means)", "BIC (Gaussian mixture)"],
        [None, k_sil, k_bic]):
    ax.plot(ks, vals, "o-", ms=7)
    if best is not None:
        ax.axvline(best, color="black", ls="--", lw=1)
    ax.axvline(4, color="gray", ls=":", lw=1.5)
    ax.set(xlabel="k", ylabel=lab, xticks=ks)
axes[0].set_title("Elbow: where does it bend?")
axes[1].set_title(f"Silhouette: maximum at k = {k_sil}")
axes[2].set_title(f"BIC: minimum at k = {k_bic}")
fig.text(0.5, -0.02, "dotted line: true number of clusters (4)", ha="center", color="gray")
fig.tight_layout();

**Reading a silhouette diagram.** For each $k$, we compute $s(i)$ for every point, then — cluster by cluster — sort the points by $s(i)$ and stack them with a small vertical offset between clusters. Each wedge is one cluster, running from its best-separated point down to its worst.

A cluster that is mostly tall and wide is well-separated. A cluster with values dipping below zero has points that are, on average, actually closer to a *different* cluster than to their own — i.e. they are probably mis-assigned. In the $k = 3$ panel below, the orange cluster has points with **negative** silhouette values for exactly this reason.

In [ ]:
# A silhouette diagram shows the score of every point, cluster by cluster
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharex=True)
for ax, k in zip(axes, [3, 4, 6]):
    lab = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
    s = silhouette_samples(X, lab)
    y0 = 0
    for j in range(k):
        sj = np.sort(s[lab == j])
        ax.fill_betweenx(np.arange(y0, y0 + len(sj)), 0, sj, color=f"C{j}", alpha=0.85, lw=0)
        y0 += len(sj) + 15
    ax.axvline(s.mean(), color="black", ls="--", lw=1.2)
    ax.set(title=f"k = {k}: mean silhouette = {s.mean():.3f}", xlabel="silhouette s(i)", yticks=[])
    ax.set_xlim(-0.2, 1)
    ax.grid(axis="y", visible=False)
axes[0].set_ylabel("points, grouped by cluster")
fig.tight_layout();

## 3. When $k$-means fails

$k$-means has built-in assumptions: clusters are **convex**, roughly **spherical**, of **similar spread**, and every point belongs to some cluster. Real astronomical data often breaks all of these: elongated sequences (the red sequence, stellar streams), curved structures, and a large background of field objects that belong to no cluster.

(This isn't necessarily fatal for $k$-means itself: a **kernel** — a non-linear transformation of the features, e.g. switching to polar coordinates for data curved like the moons below — can make the clusters convex again in the transformed space, and $k$-means competitive again.)

Here are three data sets designed to break $k$-means, even when we give it the correct $k$:

In [ ]:
rng = np.random.default_rng(2)

# (a) elongated, correlated clusters
Xa, ya = make_blobs(n_samples=600, centers=3, cluster_std=1.0, random_state=170)
Xa = Xa @ np.array([[0.6, -0.64], [-0.41, 0.85]])

# (b) curved, non-convex shapes
Xb, yb = make_moons(n_samples=600, noise=0.07, random_state=3)

# (c) two dense groups on a uniform background of "field" objects
Xc_cl, yc_cl = make_blobs(n_samples=[250, 200], centers=[(-2, 0), (2.5, 1.5)], cluster_std=[0.5, 0.7],
                          random_state=5)
Xc_bg = rng.uniform([-6, -5], [6, 6], size=(300, 2))
Xc = np.vstack([Xc_cl, Xc_bg])
yc = np.concatenate([yc_cl, -np.ones(300, dtype=int)])        # -1 = background

datasets = {"elongated": (Xa, ya, 3), "non-convex": (Xb, yb, 2), "background": (Xc, yc, 2)}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for col, (name, (Xd, yd, k)) in enumerate(datasets.items()):
    plot_clusters(Xd, yd, axes[0, col], f"{name}: truth", s=6)
    lab = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(Xd)
    plot_clusters(Xd, lab, axes[1, col], f"k-means, k = {k}:  ARI = {adjusted_rand_score(yd, lab):.2f}", s=6)
fig.tight_layout();

*(ARI is the adjusted Rand index, a measure of agreement with the true labels: 1 = perfect, 0 = no better than random. We define it properly in Section 4.)*

### Gaussian mixture models

A **Gaussian mixture model** (GMM) assumes the data are a mix of $k$ Gaussian "blobs". Each blob has its own centre, its own **shape** (an ellipse of any size, elongation and orientation, described by a covariance matrix), and its own share of the data.

It is fitted with a loop very similar to $k$-means:

1. **Assign, softly:** for each point, compute the *probability* that it belongs to each blob. A point between two clusters can be 70% in one and 30% in the other.
2. **Update:** recompute each blob's centre, shape and share, using all points weighted by those probabilities.
3. Repeat until nothing changes.

(This loop is called the **EM algorithm**; step 1 is the "E-step", step 2 the "M-step".)

Compared with $k$-means, a GMM
* can fit **elongated** clusters and clusters of **different sizes**, because each blob has its own shape;
* gives each point a **membership probability** instead of a single label.

$k$-means is the special case where every blob is the same-size circle and every point belongs 100% to one cluster.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for col, (name, (Xd, yd, k)) in enumerate(datasets.items()):
    gmm = GaussianMixture(n_components=k, random_state=0).fit(Xd)
    lab = gmm.predict(Xd)
    prob = gmm.predict_proba(Xd).max(axis=1)

    plot_clusters(Xd, lab, axes[0, col], f"{name} | GMM, k = {k}:  ARI = {adjusted_rand_score(yd, lab):.2f}", s=6)
    axes[1, col].scatter(Xd[:, 0], Xd[:, 1], c=prob, s=6, cmap="viridis", vmin=0.5, vmax=1)
    axes[1, col].set_title(f"{name}: membership probability")
fig.tight_layout();

### Density-based clustering: DBSCAN and HDBSCAN

**DBSCAN** defines clusters as regions of **high density** separated by regions of low density. A point is a *core point* if it has at least `min_samples` neighbours within a radius `eps`; clusters are chains of connected core points; everything else is **noise**. DBSCAN needs no $k$, finds clusters of **any shape**, and — crucially for astronomy — is allowed to say that a point belongs to **no cluster**.

Its weakness is the single density threshold `eps`. **HDBSCAN** (Campello et al. 2013) effectively runs DBSCAN over all values of `eps` and keeps the most stable clusters, so it only needs a minimum cluster size. It is widely used for finding stellar streams and open clusters in Gaia data.

(`eps`, `min_samples` and `min_cluster_size` are tuning parameters, not constants — as with most unsupervised methods, the defaults are rarely right for a new data set. You'll choose `eps` yourself in the next exercise.)

### ✏️ Exercise 4 — Choosing `eps` for DBSCAN

A standard recipe: compute the distance from every point to its `min_samples`-th nearest neighbour, sort these distances, and choose `eps` near the "knee" of the curve — points in clusters have small distances, background points large ones.

1. For the **background** data set `Xc`, compute `kdist`, the sorted distance of each point to its `MIN_SAMPLES`-th nearest neighbour. (`NearestNeighbors(n_neighbors=MIN_SAMPLES).fit(Xc).kneighbors(Xc)` returns distances with shape `(N, MIN_SAMPLES)`; the first column is the point itself.)
2. Look at the plot, choose `EPS`, and run `DBSCAN(eps=EPS, min_samples=MIN_SAMPLES)` to get `labels_db`.

Try a couple of values of `EPS`. How sensitive is the result?

In [ ]:
MIN_SAMPLES = 10

# YOUR CODE HERE, should define dist/kdist (sorted distance to the MIN_SAMPLES-th neighbour),
# EPS (chosen from the k-distance plot), and labels_db (DBSCAN labels)
# dist, _ =
# kdist =
# EPS =
# labels_db =

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(kdist)
ax1.axhline(EPS, color="C1", ls="--", lw=1.5, label=f"eps = {EPS}")
ax1.set(xlabel="points, sorted", ylabel=f"distance to {MIN_SAMPLES}th neighbour",
        title="k-distance plot")
ax1.legend()
# n_found = len(set(labels_db) - {-1})
# plot_clusters(Xc, labels_db, ax2, f"DBSCAN: {n_found} clusters, {np.mean(labels_db == -1):.0%} noise", s=8)
# ax2.legend(loc="upper left", markerscale=3);

In [ ]:
# All three methods on all three data sets
fig, axes = plt.subplots(3, 3, figsize=(16, 14))
for col, (name, (Xd, yd, k)) in enumerate(datasets.items()):
    Xs = StandardScaler().fit_transform(Xd)
    methods = {
        f"GMM, k = {k}": GaussianMixture(n_components=k, covariance_type="full", n_init=3,
                                         random_state=0).fit(Xd).predict(Xd),
        "DBSCAN": DBSCAN(eps={"elongated": 0.2, "non-convex": 0.2, "background": 0.25}[name],
                         min_samples=MIN_SAMPLES).fit_predict(Xs),
        "HDBSCAN": HDBSCAN(min_cluster_size=25, copy=True).fit_predict(Xs),
    }
    for row, (mname, lab) in enumerate(methods.items()):
        n = len(set(lab) - {-1})
        plot_clusters(Xd, lab, axes[row, col],
                      f"{name} | {mname}: {n} clusters, ARI = {adjusted_rand_score(yd, lab):.2f}", s=6)
fig.tight_layout();

## 4. Cluster validation

We have seen **internal** indices (silhouette, BIC), which use only the data and the clustering. With simulated data we can also use **external** indices that compare to the truth:

* **Adjusted Rand index (ARI):** the fraction of *pairs* of points on which two labellings agree (same cluster / different clusters), corrected for chance: 1 = identical, ≈ 0 = random.
* **Normalised mutual information (NMI):** how much knowing one labelling tells you about the other, from 0 to 1.

Both are invariant to permuting the label numbers — cluster "0" in one labelling need not be cluster "0" in the other.

With **real** data there is no truth, so we need another handle. A real cluster should be **stable**: it should reappear if we perturb the data a little. We can test this by **bootstrapping**: resample the data with replacement, re-run the clustering, and compare the new labels with the original ones, using ARI on the points that appear in both.

In [ ]:
# External indices for k-means on the four-blob data, as a function of k
ks = np.arange(2, 11)

ari = [adjusted_rand_score(y_true, KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)) for k in ks]
nmi = [normalized_mutual_info_score(y_true, KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)) for k in ks]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ks, ari, "o-", ms=7, label="ARI")
ax.plot(ks, nmi, "s-", ms=7, label="NMI")
ax.axvline(4, color="gray", ls=":", lw=1.5)
ax.set(xlabel="k", ylabel="agreement with truth", xticks=ks, ylim=(0, 1.02),
       title="External validation (only possible with simulated data)")
ax.legend();

### ✏️ Exercise 5 — Stability under bootstrap resampling

Complete `stability(X, k, n_boot)`:

1. Fit `KMeans(n_clusters=k, n_init=10, random_state=0)` on the full data to get reference labels `ref`.
2. For each bootstrap: draw indices `idx = rng.choice(len(X), len(X), replace=True)`, fit a new `KMeans` (use `random_state=b`) on `X[idx]`, predict labels **for all points** with `.predict(X)`, and compute `adjusted_rand_score(ref, new_labels)`. (Not `.fit_predict(X[idx])` — that labels only the resampled points, in whatever order `KMeans` happens to number the clusters this time, which breaks the point-by-point comparison to `ref`.)
3. Return the array of ARI values.

The cell plots the stability as a function of $k$. Which values of $k$ give stable clusterings?

In [ ]:
def stability(X, k, n_boot=20, seed=0):
    rng = np.random.default_rng(seed)
    # YOUR CODE HERE, should define ref (KMeans labels on the full data) and scores
    # (ARI of each bootstrap refit vs. ref)
    # ref =
    # scores = []
    # for b in range(n_boot):
    #     idx =
    #     new =
    #     scores.append(...)
    # return np.array(scores)

stab = {k: stability(X, k) for k in ks}

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.boxplot([stab[k] for k in ks], positions=ks, widths=0.5)
ax.axvline(4, color="gray", ls=":", lw=1.5)
ax.set(xlabel="k", ylabel="ARI vs. full-data clustering", xticks=ks, ylim=(0, 1.05),
       title="Bootstrap stability of k-means")
jitter = np.random.default_rng(1)
for k in ks:                                   # show the individual bootstraps too
    ax.scatter(k + jitter.uniform(-0.15, 0.15, len(stab[k])), stab[k], s=10, color="black", alpha=0.5, zorder=3)
    print(f"k = {k:2d}:  median stability = {np.median(stab[k]):.3f}")

## 5. Summary

* **$k$-means** minimises within-cluster variance with Lloyd's algorithm. It is fast and simple, but assumes convex, spherical clusters of similar size, needs $k$, depends on initialisation and feature scaling, and assigns every point to a cluster.
* **Choosing $k$:** elbow (ambiguous), silhouette (geometric), BIC (needs a probabilistic model). They can disagree — and none of them can prove that clusters exist at all.
* **Gaussian mixtures** relax the shape assumptions and give membership probabilities. **DBSCAN/HDBSCAN** find clusters of any shape and allow for noise, which is often the right model for astronomical data.
* **Validation:** internal indices, external indices (only with simulations or labelled subsets), and **stability** under resampling. The most convincing test is almost always physical: do the clusters differ in a property you did *not* cluster on?